# 离散耦合水印：真实末步容量 canary
此 notebook 只运行两个预定开发源，捕获真实末步概率并计算可写容量上限。
不会执行水印优化、训练、确认集或多对鲁棒性实验。尚未在 Colab 实测。
选择 GPU runtime 后 Run all；需要手动完成 Google Drive mount。
结果写入独立时间目录，失败和未运行样本均保留。通过只能说明接口运行/容量未排除，不能说明水印有效。


In [ ]:
import base64, io, zipfile
from pathlib import Path
task_dir = Path('/content/ceg_discrete_pair_v1')
task_dir.mkdir(exist_ok=True)
payload = 'UEsDBBQAAAAIAKyeel0iHa3MqQkAANEZAAALAAAAY291cGxpbmcucHmtWG1v20YS/u5fsafiUDKRGMvp9XpKWbRFc4eivV6QBveFEIQVuZLWXnLpJelYafrf+8ws3/RiO8WdAdvkcnZmdl6f2clk8m/ptrqQZlY6VSl3p4ut2NjGzVJljEhtUxosReIXK3KbKSOsE6XTd7JWIlO1SmssVDVeo8lkcrFxNher1aapG6dWK6Hz0rpayKKwoNG2qC48TSZrmRpZVarqiPqli3ahaPJyL2QlivLi4iJTG0i2a7nWRtf71R3LDu6kaVQVLi4EfkoRgziSlXRO7ttvU5HV+1LF+LAxVtZffhEysQa/qMh0Lv4SizkdDErydmMC/NPVRhe6VkEZhvwVX4p9UIqvxWUrj36c1JUS/yVRr52DRpPX9yV0U5mQwnMA46JQWxjgTo3PIPwZJr0+rQK6So2tIDiqmjwIp2IeXU6FrK2J52o2x7Nrn796VJE3vSgNM+dNVQswFLUVtlCvRGZZYKWNKmqzF7VripQcW0ttqlYrp+DKApZ9IVp14Itve18F8OYHVcTvXKPCC14Sb3EqWWyN8rrJhagRRirRRT0V+LPk5fX55dsFWaDI2IMXvPQts81VvbMZL1AoUBSt8jZ6qyA18HMJw4wMQq8IiDNBU8KmZ5Zd2O+FM+Dpb+Bpf2g4/YqCIHBHi4O4kbV+gX379c/Er7VcG5hVK7F2St4shLHvlUN6ZWpt7Q1On6l7BIur6qjfJ6E7W4iCEQetkBPBDIe6AXk8qZjpJEwWV8sI4aCrOgiHA6zP7nafuLs9B6wayKlYT8kntqkVTJcwrQyXsK5/XodLbPWuglVL5ep976e1bYqsCiplNuNY9fxnuS74U3SbIKovwbN9myPol/DSIcH8gOBy2UolOVSvvBjkuzK1HEkzdip2GgbhvV6jsaf7tGsT3u8nd/MT3Gzs8PYNeB15/STzPCUsVukMjt8dloCurFLUo+ZKMzmxvD+leN7KfMbVh4taksyjqZjNI9gimdEzHseWuDEPmKFQ7zsbeGt5giFfbKVZv5hJEef9J66kvrhS5ONz0lEvvXLGbo+WX3Su6pdOAyyX98EljsCF2ddrFFukjC1QkBx5ym4QRAoJIz4oZ4djrptsq+pVZ8L2zDeyLBGxt43kxE6draqYJKgSdc4WeB5ZpPX9ccG/D9EDnbiHf0RwjmPPLgyfioTvWc22+q5V2xEmB5WGRVBboSBrOdPrn+U9irGRgKei/+Bk5PVDsbyV0nCn+2PDtwe7BmHKnHA8OQiSKWbPk2YPcuy3cHYbXQaqyEoLfx/ZHAL5ZIj7nkJ8HXurHpKOIs9HXL/hgGyjZKVRHqktdc9QeQjUB7bBeyuKGYfupwLgjFPpuc7AKei5Ph+JCGGEq5Mdo+Nh82MnG6vOPstOSJSp1PmNByc93trZrKW4OE5jdo+xqNf8tNPjwryzhGV8egJklgCBBMr+XKYCXPZ7k++mopmKu6WQjoppji4H1PJsa9DRzbMRLk0tNjEwHRRumRwixZFe58Fi64mergeOLylnh+VqJ0uVzBdL+hZcTcUV9xEgrEFGKOL46eTuUSSyWuUl0KKsAbpuUJau+Lfnd1hMHgCxg/QnSxbQS4tbu01CVwRmAQuhT4NPU183uSQDQG6Lgx7m8z4eNEwW6NbUsMXsZHF+ssitHZlxvDhfPlTTjlrB4yV7UFPCiUCyHAwJhZ5nO4j5jOYe6Wq90amWZuiQmFCcvVM5gPNCMLr2Yc6tvsphfIWS/JEb7MdoVMqomfviN0hHLqh9bGS+zqTIFqgOXGTgPSq7GfqrL4lIr5lco22Hp42UOU894ZB7laTcaHOP4PC0I3TFFlBKf1Dx/DDLXt/LtBabxpgZjQEtC2z8T/DT85+hwnsNHN6Q839CGP4sPLblJLhGwGLGi4b68IOT79kmDv2bBz+NfoUxY0clMhMdgH8lZHZNXYz7fv3eCljWkV3tpmdGfFILq5EQSdOpaXIcHRjU6XVDq5QA7ALWOmOxiF3UZN68T42Kxqf9v8wKDyOqg3T0M53HRKeAmtfHqNpHdkggmCe/0dQ3v3oyh/sZjLFTsa1o6sv0ZqMcLNvbfVw7YCtOWoz7CGadqoCqFh2f44T+wBJxOUoga9o6ygUq8CR9AUU+jssntUf2R0E7UEVuG5QlCD06zNjFUCdKbbkPDjstVWLPyZvttKVRTcLuedTRxm2tkAlqC7VBgtAn20ayE+8AkKMWxZ7hszaDMQsnYLr8lP1z7J99yn4oOtrOuU8T5owdfgbHsM+/c0hHjMY67Yr3MGQMZhzF7yR8xNhUcuS9zgHxRx9QjqOHd72ID9T2VwRj4lxWN+RIDi/+f8TLVAnRLE+Dz3XBh1AKiMazp4yMR0JPqiHO4YvDqqKWGXCYTVkUTap8lwSoIItUZSvyTBWgBC8ECglYS8zhCzEATdSJN83a6JSRhYNjkJ94w5ZZbWfs2RwtBxPdq7Yvpm5f1nbrZLkDYWqkznsY0t5u7WS1M3rdX0WR1O6Wwb/8dXy7cAb9twfwoZWp0tg9NSQY4bbRaJ5Du1J3qhBvWtdblymqdXQjoLLAg1WWGB50Ir3odCRUc/W3L4PNhMhmd/OPv4Hu94+/6d8nkSroGgNOiTK9VcM1ApvVxxR1Ey+BB9eHMBbvSFi9ZOFN8AKIeLnkRB7fRjFl60cj9yikweCrX/V9rXDejb6nO7jsWqZkFc8QLUKKlwSg8GfrdEZIhgK4JhqkbNvCOme1ApMA5M/EHqjkfiqGZ/zO/bC452HxC6AUAn0o1ldfjafIl/hAEOYf+PD3cNnFIGqw268A7esSkbcpu+urtnFcIiCwiL/9/eQlDQA1rcEoj4XHjwWgEKYN8n09qgC6B4wUywx9BulTEcxn/Daf0fvywOrIPTlM+/LF2p+Qboi4xTI/qC75jqw9YypLmVLjdApwPAsONGmzrZ0GBh++szeqmBmAh6YsEa1+amV0wHfRb//1PQrJGoihUICapES60zhqJt69eXuUcEe3wkcKnI9Fryxbx9uADsrXdDARBuHrkE6siiZXTtZd/owvuICi4uEqNDq6rywTjVZfJtfLo/kC2wBY6ALx+GaRFYpgJ0yfQabTmoXGrVI+1wGX44TUWxLS26qCtePbWdjYrNiaNFmd1PNaF81wZ9nja1LoPL4eF93/QbV/whpqenYk7bXrkXLsoRErJRGsw9uakBNjsJjfb/tLzcd5cz9e+fCKR+x4ARLao7cESTsgPM6ztzPBfGaHBOfpuHvBgNxh+A6/d0Zj/cmcU8q3Vc+Lks8ln3fvny89nuLrBr8BDP8AUEsDBBQAAAAIAKyeel0kCD7FgAsAAP0fAAASAAAAY2FwYWNpdHlfcnVubmVyLnB5lVlZj9tGEn7Xr+hVHkhlJXrsBAF2Egbrjb1eA0ngje28DAZEi2xqeoeX2c05PJj/vl/1wUOkPLEeJLGP6uo6vjq4Xq9f3zWFTKVmb9593NVVcc9S3nAM3DOVtkJUsjqwTtG3vhKMp7rjBctlhe+Ua3GoW5nif8Fvo9Xq95rdYrAteXvNRLkXWYadW1bVLBNpnYkWW7X2Y2ld5RKLtayrnRIa5FOhVLT6gKMUvxEZ0/W1qFjT1nu+l4XUUijGW4EReYOTWCb5oaqVlqkCSXGDEzKhRarrlsmq6TSordfrVd7WJUuSvNNdK5KEybKpWxxYVbU256uVH2oPDW+V8M9gUos7Xci9H7E/GIhKoXnGNfcz/1N1ZY9quL7CCn/OOzz6RUoeID3/pGXZH1V1ZXPPuGJVY6mkdQf1QPZuwR5yb++T62LbqylpIdg220ID93WnV6vVPweOI/e35BU/iHaVidzqLoG+EpAgaYQqvRJZV4h2c75i+EClB6PgmPVTkdKiMbNuV4bZh8eVGSKyNB+WUHGRgA0IfmuuRsNb6LJsCrFlB1GJlkM38e91hedWgFSVZDLV8Ye2E44B+sgc+tOhp7FhcczOhmm3xDMznaBPy6US7I+uIgqv27Zuw/VvNSxHX/GK4XRnxET8nHWVuGtgNbiW5ZWsB4ac81SsNxPi37CX0Fp8xv787yuZ5/CNuno/EZMjoWCcUI6GTxlljoUTHZF8X+dQ0h2cAu5QyM9k5TnON07XyEbACkSg2KGTGa9S8Uy3XZUaw32WFjhuStCL5SKYeE5wCaVN2ICv8PQq3ERZ3e0LgT/KchI+30Rp04Wbi7PL6ENkLDPcnDjkSiro9N6Qt3cfCBsis/2tUF2hsdzb2leYTv9vaj+j/5uvsSKDMAqsWJ6iphU3iT2Vto/IMlHApOwyiOWEMCw9Iwv792lZ0AmOrnWoqduRVL336fZ+YP9eiiLrjzbDxqiL0ZoZKS/xp5FC1V2bioSEB72ECkBTZcr5KEQD6GRXXHGt29CC2pYF79++efnrH78FI1deckWEGwYThiZx8U8dHEWxX2XV3T37pS74nuXAb+44MNbACllK7XyRuIPDYldmTu7KLXyMl2J26gfLvD01XzuC4i4VIoOzP7hLPTL3xx1QF4Rv9lKR/XFXjNwFt54Bu8FN8oK3ZUgm56W1OaG2E9oakznbHI+f4ATcbqBOkgpkGiKCeS21dU1eRuEnRPSTBWLfJoKw6+KGvB2RTlTa4joF4wMWUxCLippnKgxp/zMCEV2ndRHRVED7eZaQtYTuelhQNlotbrYiV0t7DTzRxuDS2xTxDluErbCf2HNGVjCM/OyYvAiAUIkjHFwe6/xPXnTezuxGWICSmSAc/ox0IkOeUNRNiatDQvCL1qn9G/bWBFr17FbIw5W2uQaASe4JcgSg3MIymOrM9hugdWZw2CKwj+p1m15ZFRP0ZyZQiFb5+VHseOfQfexVZnuUdhmPpEr4DZfwCYLnLzsV0q9fPr56SblcxF7VhpTq9siOdIdcibNf3n18ZoB2lyO3I2MxjgadFEzcQEIILZGTBU0gxbgGHGzZtbgHkLKLMOivEsDZ+4cEiRddJthsWRjollcK+0u3bPw8rLycBvtZUhW5laHjYsP+FnsDAD+XUxhfkEe+flspDQdjD47EYxw/jCg8mktSiAXmS3H7hegPC/J+ZAzSBqp+LiqvEexD607K5DIEEIiLSX0d/5sjcGzIvGyGWuPrtpVQCY5XcMqKkgQKAKXJbDEKmmmNoGUOKEQGTMb5FxSGQofLMovzAKa8e5DnZ989QtDWm2L7g9AJjIu9x9BDsudKBJd/l9tZukQYg0S4U3EAq0lgGcHGSEd6sqR/AaQ1jhA6x704H9zzcmO92OsP/Bp2LQeOkS0za+NhG0J708Ujiz8IneBaElesAOmAwUV2rZN4c3IEEv+cJNveouOADHxnDL/Pm7fGO5AddUpXKDqC5UOsUISKrQ62NNAiTYzJzCL68hhoIia/mThpCDUCAQ0ND4BG8RYBDVhmSAdU6IW2hZjBtI5fgP9KUbHCVSqlsyEXbMwxK8cfsUPJBjFU1rhVXcnUTVPqiDnKtudxyE0uYFFEqJUgE4Ln4jELe9w1WZrMgsut8Rkj+umkH8aSRYl+8WN1mun7RjiN5ggm+vkPm0jXYUD2EWwmF4huPt1wQbOzXd+9GJaSKYO/jpK6ioXDzq2lMoIoPwINJTAfqlo3U6yxhCIB8A8potn8JTm0CG2hVdQY2gy5IQ+z4qI0MDHiTXx2qgjgnp+dESocM+V3WXxKFEoEWv3diydRcP1xVNs4JugEW1kfWpmt59wunEz83pjCOjGVvdo9Jw5GOhgv/JT05b96msVXplCyLO10vSOh7+v6mpW8aUz9q4y7mmg7YrdrGtEme8BIRsnHxeVE3219S8q2njtlQhS8UYtus3MedVxp+h0/x5MkBECZuFQvuJxXod6tgb+6buAbnECAaoOAdtLhyb4DfzqY7d1j9fVUdPXthUOTngY1aKZ7R+jQy6mSOjEXOwUU/jMBiH5wAGfYOeC5AWdJKUoUfURWqyMq9LmV+uq4hChlFc5TuEGAW2ZKz60p2k6Keed0scFnzmx/tuVaVrloKQokBE8h8pN5A2TqoBvqwLipZfL0sQkAsw4QGs3YmEi3IBcYDja+3aPkwtxfh0pf/ycq5YXoaU6HiYOhP5BQvO5XHo1/xcl9/W3l+sY/hzZWxw6cI1SQHeRL+YYVC/0LLjdzC4FXwZhCJ2qT3j0cNSyQPPrmAuWRtrR+XNbKMrBkBjjcGV/uYVpsXc8ZpbLZUZh1VJZu1UTqijcGoMPnZy++37Lvz/7xwwlbXUxcR5ht+NzBXJkl+uCoPy4wanuBlttxdzBsfH+QHMDbQlHj1sl14RAI2RZVYqeoRl2TUeY3ZJ9Wu/6RrM5knCOlP2VcKq0RsgMD48zAONuLnLp0f7z5F+NaI21HDYbhT1AZNaXBuzJRE6zOOa0gG6Df5wS5M7IXBZAwKVi+fjhm9jFxLeSkx4Ooaj6vt+zbb71Fzui7rpksUUso6owZrD15AqXbUVMdFjR1epNXnckX16fyRd/19dnignuNg2OEOCqqzG3DiQAhKN4sWVR5fdvr25UFJNJCADKg5tHu+K+QpH4LKojX5gfYQwgryNQX+rbzk3OUv7Cmrd2CoueBMr3QPG2Q8lOVkCSP5+zBDD0uGMYX4rATvjtksnPWnhkxeRHcYm4ISkupxBB2lymYMAoqNaEQqrN7LXzjsA+4FP1cpO1XLkTbhaBPH9h4NbWb+naxsEBYLDp1Zbvwx7nPNO8Aqg3GMJfNNG3BbpS34dgWTROWBm1idgSLLpsmEOvfdvSpQCt83NRNS4Azn8ibmfVBNC0cFhRNWRACJUrBj5iabhnspUdS2HfJXZPb1LVEY3ADd8i2v0DSsx/7oTkcTi4Un7zm5HoLy8yl59QzkdoKLZDQeJ7LVAIsEgBsMuxFJqARYxPbCAtME84J7KdBG6b17VoD0Bs16hcqZo/nlaTagZXyzkTdOqfAq+iVyx1o/YfRi8YfTWDmrKmbrjAZCTM5uy/IWSpaLcEzpSmDdoiR86/W1VgSIyDzQTYemSMZlV1ghxes5C/A2ZcAR+iumeHNsMFQMkufRrppO3C5pzxQfhqxjoufEay41xT9K8/VCpbiWSKXDpKkpHI2caBg3qJS18q/UY1euq7pOzPjWxTmIeJZlviuahjsdjbUBoM/jaDpxBbTTsIO0weQ1FnKRM67wjRTrkTRxMGLnX8DUB1kJURLxWXKyVF/RDFtysa8K4rjt+DOAKlrRSm/Pd/8EAeqF5F7q9V34V13/BiE7Zsm5Fzh3Ga3SIK9uQbnbGywj5tZwDdvqAZqY2ObNYvf38PDy9d3UocvNqv/A1BLAwQUAAAACACsnnpdL4tr3MEBAABVAwAADQAAAHByb3RvY29sLmpzb25lUl2O2yAQfs8pUJ7XLtiJs94eYI+BxjBkUTE4gK2Nqt69gNl0lb5Y8nw/zHwzvw+EHBfvohPOHN/IUeogPEZsFtC+2VgjYAGh473paDcw1g3Hl6L5gIBZ8E4JWEneGZDkM8GkTWKTL9lPYh2BNboZohZkXk3UxZtgEGBS0dndcXYSDdcym85aeBecij+2WyO1UmtIvEbHj9t3ssdNZyBLenUZBzWdRsTzeewvVIqhf5UITL4yNoBiPZPj2O/63RN94Fv6VAva9mNLd0L0YINyfn7inNrzpa0Z2HXm2ir0aAXyEHEJicIoLeh11RIKkObMUZ1bWq1XK8rg3EPMCHsgj0e5jPelBKyMg8jqk9ttA3zC+m7HAqLk076WfVW0tpLzLq0N5de41BH/Zfi0yitGbiFmlLasOxeCx6+h4uIL8lxXtU7ZExCEq60hBF2PIYJP7xBnzf0l3UNMF0NAfGjcUJLETzFoeyWL0zaxPFnWyeg9IiK8juj/XQl88pQfDyiclbntS1enzFBwq89dPNDhPzBX+25PLLqFpzVUFXrv8lhpQVjwZKK0n78Jh1OdNwW43yB+olgj5rNVYAIe/hz+AlBLAwQUAAAACACsnnpdaDKuzhoEAAC0CAAADAAAAHNvdXJjZXMuanNvbmVVy27kNhC8+ysaPmf3A3LLJcf8QJBDi2qNCFFsLh+jFYL8e6pJjSM7gAF7xuxXdVX1329E78Vpkvdf6f33LGUlbnXVLDPN8pSgaZdYKWXdUy20aCbHiZ2vJxWXRaKPD9IYzu/0h5LTuPi8c/UaCW9jC4GKtuykEGexB6XtMn9//8VqX3lR/U98xBe/kZV2knn3jqpw0kqTFD8L1UPpWH1FlpYKihLToTpLpM1Xt+J35SnIyN2TlZ3RwPVoUa1T9vMDCbKWYp0zlZqFd/KWLQgvJyXO2y3HFJrQ5N3pgtiLPjE/2MdSEZMYX5eqUeiwYpZoqZKjosHSYvCPtd7S9c+rtiJjgqxuO4ELI1uLs2RiQPeU7Oybsp23WIw34S1XoFTqQB4pFg0zUHvYOtAux00A2mnw+DjrccvwCFwKTXoE0gUb4viwzcQZk+3YzYXqQFlbxCBV0y1B5Jz16KihyOHrSqhOwNVt1McqY36D00alDKTuK2EfrPNJ+Yp/1bP/jPqOwz5wD7zd93lKCKg+S9nwrz2RD6HtPnIHw4BLgCBqlUl163NhQxKdD7csSwDmNn9ZRdIo+cgA5qQVCTvZJumFfKkM/u8GhW38zgttPxowQJ7UcgILkPeQXAaXKrAtoJLJ58mvbaNiWG5JAE44GJwLXCGu/YUIypqGOENmPoANlg958TPz+ZVTr32Cc7OxUVsaiZ7ykK6Ja7EzqI2hOoX/p5WoxzdnzDMF8uSNvli2kXI2OsEP8uCYLf8TFKAEGtMHmq19FZdkuct5Ml30+oGPOxd2tAMEMSweAGugcFr00p2oKoxEL3qOwklSAsL/pYgkMJCTtNVZYTgTF2TsBUHffDGsUykBv4puwL5PzQ/tLteGDeG6gvxi3DdQ+XKR1KYATyo/Gpb6KZ4Hm5wGzUsLZNwrFOVnxQwf8SZFdIgZ4313B6PO0gGQgqrzgK8eNm/flznERSpQBMK/RWOo3uTqtyEqhUt/gO84gnoAxW2pN2k5kma33hvo4jOn2L25NQjfexgFvUDdKHAp76L4LfzpNdhT7MjXbi0f5gS99D4uWcKZ0ANs7Y6emQnSL76sL1cokFu/QGz++qN5UGTlPGn+hHr2ybgl7FZq+5TRoE08SpeV5y65Yc/o7hbrgkKPTy/H2O/Uqu0gnBdAPm704NxvRp/387Z9hJ4K/hIJBLbV8BJ3hXoGbvBD877rLt1r2+WElncM+eymFbTfIdPUN+Mm5n5KbHeQrgPmVtv2MLXBqa7OoeTRApiPE3u/GWAVPBzKi3PB6ZbLs7vz09ygBWAYhbvt4MRx/nJ0XsfYLOqKLhCCYY/LOfd+MKS8LC55F+12f7GXLuBx9xbwDOE9FfZfbAvX6TVxDMFjI5md1mqMrWAcEv319s/bv1BLAwQUAAAACACsnnpdfniZT4ERAAAIHQAACwAAAFBST1RPQ09MLm1kbVlZc1NZkn73rzgR9TB2gJHkqqKpmfADYyjTDdXlArrroaPDFkaACyMZLwUVQUxI2JIlWbKEkVfJi/AiY7AkjDetjuh5d/2Goe859+qJv9Bfnrxe6JiIcnGXc8+S+eWXX6a+EmZmWeaXZHrJKlSM8ppRnrL8GzIZVsVdGS9+ri405vZVfk/GE1Y+L351NTWp2XWVWcabNmfb5VaXq7Xt8j/9L83UpgofyIMdmYypsYSKRWQ+a+WzjWxZlhNqctM6SmOY2l5TM+NGZV+GKmbltay9lpG46HQ6Ol3uz9V0p+ueMCrrMjkpw7NyvIyPZHBdLWbxxedqzDiMW6M1a3xLRjetXMg4LGNBFclZ2ZgKz+C64V+wjsaxTmNjxjjcwsmM+oRRO8LuZH5OzZRk8L2s+kXPk77eQd+Q78Gw49enrff7HjwYGerzeVv7hh897cFCMnhg1F6bkbDKvLdyAdEzMOgb9vX6+i/9MuTz9tAC/oiaeGvUF1U8r+YKPLu4pmfyDA4JNbkuE6syETXTu7g2VwKYSs2tyOIYzZ/ZNCoVozatZopqMyuXJmAMtZQxyjkztQyTmNm8zC/AIjgyTV8oqdmc6Pix40fB43G0xlZMrn2wdslc5sKYir/BK2ytqemrr4S5kjLf7huHFVldgc9W/RiNHch0nT0ubnf+t2C30/zJDZVIGvW0qmXMSEnWA0Zl1UyG2OAqfUS3C2MMEDJj8pVxlFWBAjmldmQcLcr4tFGLm7W8VS815oMYjIOocJLhAztZezMyNA9vwrYqE8F/RuVA+qu081W/2pv45E+ZqYq1u4kLa7dEgIrOGYfbhJpwEhawajU45Z/+ACMUFzKdbmRL8iAvwyHc3rwlGm/GzPysXBzHzrElQlE4DSNjkrtX/9z5I0bdudrR2uEbGOl34+bu3R/cg49xcdtz/1mf9yGuhj3PhzsH3X1eAcNYR68aLyvwg8zH1PQuY5qOuz0HH4qea//p6PDd9zy/5h52OwY9Qx73YO+jVvfI/b7hIUev52HrgLtvsLXPO+wZdPcOE8T0u1aKHZer7bKj59Rh+Vn1cRqAUdNFub1hHOXV9mpTk0yXAQSVeae21+VkSCZ2xA18MtDdJwB/lfGblbB4ONJ33+3t9dDuB0e8vW5aCTe9/e4nAwLwEP2+h9iRgLFlbco49MvYDCFoPgjLmR8qauOlOUlxoyJ+8oz2pAwvytyEOfHefDfRGB9XKwcYICeX8VAul4Ek8bx70PNAyMQWnGWUJ9m/8GBjPMH4OYtZDuTtVVlNmOkd9rg+uiok4FuZmOE94ESNd3OEtcM1RHBjcamR3pf+ecQbJuMxZvqQWGZ2H19Z9VFZTJi7b2R1mraX2WRPWYEUMGJlEWdBqzhKvkun1UpV9DRfuPgpPPVf+Lt4oUUc7+p4L9Ybo5uEmegmzZIogFJkrNyY3YWVRc/fWp/0eZufOp0Xn7pcLRcF37lw52z5O7nQhkc4JA9H2VZyflMGwypaE+7/gbdokVCcaBGHLaeY0oi3MOAe4SuxpVIlHBqQZauppQXjsC5u/mOH7kvLeMT0piamzMoirOQwauDJEBajyM4vmZNFbVKjsiZud18lgNj7StflaAJo/RSaQtxfwT9X8Xcdf3e6H/doWAQ35Qc/2AAT0l7D86LnONV9tf220wlT3Xa66P8u54XbLpc+sMalDO4btXkzuq/8AXxFjimUGDEyfIC4s47mCW14iDDPLKsMDjoH8Jk7dVAc2fZ4l5bRJhxlsuNZwHdEVtiZ/sC2yVyBIS/XFsiQb5bMdFSnlVVrd1m+jsO0MrRjbgUADUzd7oRpGdP8EisCT5QsSvuwNCYzqnMEvowfMBIvjndf0FaSMZk4YIDZjBIMWPlDYjVsrpwy3xXMl6VTdKtoFKc0K7vqw0srt0qujwWt1SB4iwg+U5bhfUxL+0F6i5tj++dtpH1mLxMLqon3cATQKG7e+hRZOy4LORXjJ2xAcavrj113BM6OMKMRe5iAGRh7YvrFtvDEKgSQJa1cRM2uqPjyWfQmXlr+UbLudsSqB2X0bWMhgV1iPvEzwvmlcThJ+W3lECagQwJPT++5B9uv/+0pDASaG/47BeP2KnyI1C4LVTgGiqCpqedac1e31/PsxYsuZ0s73dBoutMXLReu63/FtWaa8MULRMan6OxA9y8tPfgYp6FPfP339fcY3EyjL/7cgvFY+fxwJo+IH3JB+SunJrMzAOU0IBochUfGYdScjnEyhbfMXJFkw97E6QhC0E6WQFIINabWZekjMgjeMokQLQCD+QWb0ErL9mDGYHwa/MJIBK7U2LK5vN7YCajVKdjfKG2cEHwnsLhgywJ49GDd3JwgHlxeb2pqFRTBmSJYQAYWREfXXwSlAX+VB9D+lsYa/hXmPztMdB7nLMvqASdkhrBRmSqoWIC3ovw52ker4LgVX7f97+zXbeKu77HHK8zaKzAjjua6LEg1aY1EdDOOg68A8UiBOCnRwuEosQQpol7fyEA/MualfvdvvpHhZu0RNgyxXLquYq+IyUN7nE14eaDSnJ4/z5ZMvUhWQ30PvZ77CFpcU4yCq+E7hJxwIns5hVneOJVEsloGd1obAQSccF76Tsh8hPxVL8HRvBLQC8lDNB6e5aCgeReXZHQFxgDC+YIZj8mBFEXyHW+IsyPYmahcbxRv1cEB8OBgNQNVBMnF2usMZby41ps0wTlFONT7yHN/pN8zSBNpb55kZd4qExsnDzWxocJb0IE0PStDDUIdjOskEgJv5Foc4QwxBtlAVqlodtT4YtRQiCLwtVxUi0hL8VMoktheYACxtGBgIe8hh8Mqn6thluYyVQCziU4AkqRbYvVzNQJRgsSmVbdo04DhXFD1y8QrVU4SZ2t0i1631z34G1Ew40JvrpGatwoFkvjIh3oWeI6C9+v/bzIiLf0a+tRcK8NvYsgDmCDMsQuuEpiHYXPQkZVfa6RDtGQhgdmsoyQWOI86fAgelUmSh+RwVtBTML89GP8RoSKEDjbtJxshMz1zEmM6D8UiLGH4IUlvvdU2QayqnyGUSfcUompm2yqscylymjFgXaOWoSNTpIRnUNMgK4DIoHBBGygqOCuMj+NYwuV0CjgJ2+34vrP92y9kXrtL37q9Qw98g088g+L7LhcKssBffxJIRag7kI3w7Os2Ck+NBevNO66+bIDMFMFeVn7brIzqwoYYxEZvz19/unZSHN05we+loWHPQPNwu7OlRyiw3+TBqQB1nO3L8e/ik86u/SCeuIceC6LgyanG/BqFzzfO7y4L6FB20QkYo5CZLG5o7yx1bcbSRRVLKfHHuzd+ElxgYQ12GRIXBwE/l2s5pHd4mamO/YsAIuPqooo20Xblyh+giK64vmsTXKmdE6jiuNzuvORq+1Z43cNMlSRS4LDDaGM+iUDAc4wnV/spHXIFCtTjrYoUISzPQwAsfwpWFXmnEgnaU2Vd3O26jXXIxd/rK6dLACiQLZ/8GS7UML2sz0JQIzABCiv6gcs4rpk++Re1ZqIMr17H8a2ZzquV8VP9RHZHKIL5RmtUS+XLKMgbs5sqlwYOoSgBlttCrhbV3CSdJ58lXcJwgDtPdBCUKosgm/pwSsoPpX2KUBItiRlYnA04XQSe5doY8qdZga583N+MY754gYVakPJv3kLO//kHpPxu70h/Pz263n1DHK8el7tRNI24+7ufuJ8339AZBnKe1KSuAMQNcUrNdk2QqsjxCiDcCLymJMvFk45UozwBxU258t3EmX7VNGDne9gV6VdjimRrOY35yZZ4FfMz9Z8pPl0Sn49a1ltkMV3/IFupxWVZROm2hUwD8iTzpFfU9homhx+MUtLKQfSWqFjS9tPGahXWxDrf00r7H0nf6ETOPj3x5oKVL8raNMFNnwGfyMQsvmKU0b7tqWkSe99chM1hi3uMGbvBEEetuUbQIkVPZGtuVNT0qh1s3ArixHa6OUwq16gLRPVVvkjoBIGCWc4diOMbiMQy1tGcnQT0bKSUtK+YLG1xUKkQpstJThEkkOMfbe5mmoZoD2TU9hscW73PcqjbigAhoftCqKKY5hyoMWR5w0HZDVoqFJehKRkMAiVUkeh+li2J4tNY047jy6BaMzdFObqQOF3DfvmHNn6rl5ywajXKxNyOia7gFsFCBoYcfTPfmFomj+u5WaTh8OJWn3fkuXD3uwefUNOs9sqsZHgqneBneTzmAYGRM8N220tFjqxC2cxSmYdXYEJYgQsb0fGXa1eFDAUhI84SPGrKBYI2YuVchw95vZGtIHGamQlWAcjmXKNRf4HRmizAjLoZ54Ys3bKLztl9UoQ666rIWyr113JWbgzHpwoqNqOyIbVEVQ6jjPI78nNmkysdyjPs+Ill6i5qe8NnMlyEnTlgxXHV8fso1cJ2bUkyVEWnZe2DeEZVkPh9rN1FFA/NhvKu51F3X/uw2/uo+dn/7fmbmz0oDsJTx9UWR/PvoxeO91paWqi291AdLJAOezzeXt99T/OlS5daoFqHPd7hIV37iqcjbu9wd6/P+yvlK9KJ4wnqkETi3Ayz9QDSef01nU33OYx6jgPEcYOr7qNFilJdR4uudtdlcj+q+iO/2q1wEQzWZ5+wLMCARuBIBuM0XICFrGzWfL+B0hPHN0oR8vsJX7FOFT132rs+BSr/KB2viqHufgET4O+XHkqnp50dO8UA8ZS3UgJBKLowR2+/x01toa6+/n7fM/Gnruud36IoCVNLgAT4yL0hpGwS9e1tQAbALTrdI0NDfW6vuNc/Mih+H8U+SWWyE3UA696Y7iTZDYgAq3Dme2JmBoD+BNkOT473HGZlkaCp+3126ZuIoo6iRpKONXkUJKRmNpFHKQzp+NRNUHAIl/mRuJWsmek5u3VQec+KFewIPqLWRz1IcSDYlqfMTdVfcNNun8JoWglpB7JrWDawEpalIBiEjnSwyafl3iexgy5+HWAqKhjDITAmkKSJP2cnQhD3TLGxk+EiAtFJCDhXyMsDIiBelfamqxkEjVWv46zEwjW7UQY1BDNzTxiZ2257shmSFS5RyS26FrB2s2YsfwoZUsabH/GhQy0FZPUDMa8+BPdJ1Hy+EYhTq0MTq636zymV8/KQAw+C0sqCU1+dsk2bo/Nr4htdEFNpVS1DupnlI6ojIlT9scCwtWVyknW/brxpyoHaz01waZEgzWS7AheJgm4UBljd4KgkIiA/jrIUJslJ2mR9UZYTVDd/47jisIUZq0Ugm2KXfwAIbtJZtL9Ze4GFxOVvhKyuUO9Kw1SQAhEMVodaycrxkAMbsApEyzqNkTAgSk/nCWyZLeZ2QkRmS+2HsU8idj3/aR+fk2pjLkwaYyrGCsHBR2LLnPVm0nXMNTDoezIwTMGTC1DOtNuzAV0cq8k3janwWT4/6aX+/EPz45YLj3WAbq+ycMbB7Mf/Qc8ZrtQn4IbP1Dp8g+rlvKhh9sUrVjfUV0vXSRSymNONMordYI5qyC/6WAGSYMA5y67TUPnyNwXdlwYRtfq8/b/RN1rzf5ml0iry2ijZwvaTP8UNTNZm1PaGnZfiulG7QeIqGaPmrjYLb4JreAcysW3rm7ccfMVkQ6MSMzSN7g+wWAOs6MipD9z2sXstOrlRNOlXfGv3h5eXVXoXafCsy61DmH7oyVLqY6lBNa2OepZuoDdrNKNLLMp11KNAvafhynToQCqHipKBhJmrONi7jn/7QQSVIWfllXEgFERJv1ZNFr/4MU//PkMNj9CsUZvXFa7unGpTqtS+mY4SoCHocKYoPB9Ui+RzVvwO3oxK1WW+ptFNHxOZ5ArmboFK1kJBRdchzNRkTobnmT/IJ8UEY48CYq5uvZrFAkZ5kr7djtDPBAc7ODILdipBMcmrKDfbdbUfloXqqeSR4ytGjQp1azUgI1s6N8dlpaxL9wVVSAFatL3oLPfu1MexRmbH1oTRFXG9o0NQuxfAhA6cfItCXm+NWrIsKsmc/wJQSwMEFAAAAAgArJ56XfPbPG8CBgAANA8AABAAAAB0ZXN0X2NvdXBsaW5nLnB5rVfPb9s2FL7nryA8DKUcWZUUJ22Detil1x122EUQBFqibTY0qZBUXW/Y/773SNG2krRpgRltIj/xffze75fZbPZJbYXi3Ai1JY5bZ1OitCN73XFJtCEH5rjZM/NA+BfRcdXybDabXYl9r40jatj3R8IsUX0U9UeEudoYvSetHnqJyOO7P3nrmNpKnpI1kwzAusaKrYJL10Ixc2weZEpa1rNWuGNjeKtNlxLJjnpwV1cB2pNZjfdk+C0blIDXVx3fkM0gZfNZC+VonxID/853gkmOJfdXBD6PAKH6DHC58SeTIK5AKL429KSWsQuMbJ3UoHj+3nIpLQ3AHsBwNxhFHoHO75cUe2bYnjsj/uZ0tjGsdUKrWUqqRZGlZJHdpCSHB/xdZHXibUHtxhvkmJAN4ECsmLQNU13DvwIGuMj2WllOI+RoHloULGTGsCOtspt3gF5+wB8F/Chu6iS9fF+8B+myxPe3+P62jga1DpBOocswsmcuF76TOiU7MbonW+tBdda/8O4BeeRI5oTCQbE5S35bkZxwaTlZSJ2cIvRiNGMg/SmwAN0ESZYxa7lxDZOylRpc8pjZYU8LMBN0mdNyVfBFcfuDenniL3yqZ/iBmS641gpFvQfBMZwWd0kG4dixntNlSpYJmc9JGbR8mnjPoPZljsX0wswKR422eNSrVDmkRb04PRfxuUD59em5CLrBEqQmbLAFWYIx9HExTfZkHqgkozPn/trn5rKuE0584TGXvKXLpLpPyR9a8ZpcT+UoTMl97W2/5MTW9jtk4jVJQj4SvH75c/WDpTMtG8v2veQGEtW1O25DEjVCtXLoIOi+ouzrVXMbSjIrn1TLEuW3vlj+/zp5WhuxNKSeyD9GeZ4FdcMONsIG+8eq8cCePwSq0/sM3MQGCe1DbWnxYQm2WXDsqihz+IwpC7ycDd6A5uy/Un8FBj+v55IrapLrk6ioU7IXCsRbt1t9OBcD+O/m1aIL170Nv3yaAKnvlv+YrFmeL5Ox+fvIr4duy52fK02r1UYK6JNOcN81FQgETAtOx5C/FrYqe+/DD5mWYQe9qV8MYrgTiHLzhUmKpFLyODCJc8zX1qqEHOW9FVIrJD12hrE8AOrjioC4IEAzgD5ICr1wlC+h0nxdhkTVdlKWPimrKpQCch0Loh7ZjgnQRL7tTqPTI0xKAmGk+IzamJWYlpfiSxzgAJ5uYVVQ8J9WZ9xFfMT68aGq8hrB8u9g/c3BYZRCX4NxVCYvah6E28UdwDBhuaV/MTnwT8ZoM8Y2xvcCGjPqjOxLQjEVb7hMo9MWgmlz2lYe+NHG1BnJfzt1zuHwfQQ6BRHW982fMmG6KtEZcIDGV0yC9I0zMJJ8KZ391jNhsKrDTkUnIFjRljsqyAY2PzxJhBo1UCLi18QD3pSxgrSfiE/2NnS3E9CDqgy3DmwPtMjLJVLHmYywKC9vJyRy6GsBoHoD3GEjbIa+5+ZNjZVQ3M1R4xpLYTJfTvsj9b7OiwT6ZpHdXYaUdazHXGx2ooNVFnhuuMGV1sfY8F6ylu85zAm41PflGOpfyCdYdbmB7CaMuINeeO/0RrccpvVhB+lFHrhR0I+hYHoONQwdWssO+iiYIrAubGjT4y4MPcg4rWVo/f107NzhLvbjg6UPJ63DWzC4wkIU4wUZ0OyG1tEwpTHtwVbO3GqZBMU1s9hK/rH3ZCM1cxg6VKJ95aFsUkPIMQUspkC459/QB9kBNBEAWkB/HBMq9EN4MY4mnz0p+YzaFcVFBkjQEgdDfU50xRGrg37tbfS1Fu7COkpO51ptL7Ajvj8JIwgSAq8BYhm4YH8q1/iJUzbL3+E4xfrwqgn5lZTjDprly4nO8120f2EXnfC7XgUq81Mzf+Ek8oa+vEbCPxCv8okpPugx3sGI5+8rAZVnq8/4hwve9ewEuL1yA24LEOfIe/5Y4eH6dDoEGs6e+1IIMiigwZNY+E4Rl8Ip5+c7KkbAu8pH8cUI+kCBIb6PYfOAWXA29UECs4gyB2Spt/75LeZlZevkFWgvhjXrDDmyBGRoOsHQ67CX/gdQSwMEFAAAAAgArJ56XRY0uz5jAgAA0QUAABoAAAB0ZXN0X2NhcGFjaXR5X2ludGVyZmFjZS5webVUTWvcMBC9+1cM28PK4Ahv0lAouFBo6bkf9LIsQrHHu2pkydXHJvvvO7LW2Q1JaCjUB2ONR2+e3rzRYrH42IYoNfh2h13U6ECZgK6XLQKF2lsPdyrsoLXGBxfbgB1ou1XBV2AsDLZDDXeotrvg+WKxKNQwWhfAxGE8gPRgxjk0HgL6MK+Cde2u6J0doFN9Hz06D8d/P79+mkLKmu8zr5zaylG2KhyEi8YksnlDr4zUQss7QQkhOiyKXG2Q7haaY2meVjwaFYqi6LCHFJx3CDkJIUI94UjTidEhsdqj8HIYiYKwMYwxsPJ9AfScJGueZcxIA7HHVrRaeo++eVs+3sc9BhHUQCxw9GxV1zkh60uok0Y8oPHWsfV6fcFXFVy8q/mmgvXFJacFv56+KUaLK77ZbDLGTnnafXgKclXB5ZzkkbrZwPVqWuH9iFN/m3OKRI3NDa+rGbeCLZL+kr6bXODLvGYl6WySlAmepVeZy1mntqlRTwpMfyebPWkke8gsk5uO0S53ID3Uo6jDf+OcOkcGewwOTfNwmPOsjIq/CYhlXpw8tD/6p3pQ+Dyay5iRJzMqs+UZS0itW209svnI6+Xo7I28UVoFhX5Jfc9n5J2NNxrpDN725Ph7tirX9Yb/4NMQsscnoUoE/TIqfIC6BHiTfAZBKu1BOqRZJxGURhP0gTQf7B67l5g7Jw8iq3AqE+wtWTCxfk6FE9VnJpOkJGXRiwHDznZC9nRDiZ64JYP86zRevdqUx9vDSUX72Ldo0sx+ds668uTDv9r3lDrZNoHBORZbKvMrD+DpCj4ecvlaM/4BUEsBAhQDFAAAAAgArJ56XSIdrcypCQAA0RkAAAsAAAAAAAAAAAAAAIABAAAAAGNvdXBsaW5nLnB5UEsBAhQDFAAAAAgArJ56XSQIPsWACwAA/R8AABIAAAAAAAAAAAAAAIAB0gkAAGNhcGFjaXR5X3J1bm5lci5weVBLAQIUAxQAAAAIAKyeel0vi2vcwQEAAFUDAAANAAAAAAAAAAAAAACAAYIVAABwcm90b2NvbC5qc29uUEsBAhQDFAAAAAgArJ56XWgyrs4aBAAAtAgAAAwAAAAAAAAAAAAAAIABbhcAAHNvdXJjZXMuanNvblBLAQIUAxQAAAAIAKyeel1+eJlPgREAAAgdAAALAAAAAAAAAAAAAACAAbIbAABQUk9UT0NPTC5tZFBLAQIUAxQAAAAIAKyeel3z2zxvAgYAADQPAAAQAAAAAAAAAAAAAACAAVwtAAB0ZXN0X2NvdXBsaW5nLnB5UEsBAhQDFAAAAAgArJ56XRY0uz5jAgAA0QUAABoAAAAAAAAAAAAAAIABjDMAAHRlc3RfY2FwYWNpdHlfaW50ZXJmYWNlLnB5UEsFBgAAAAAHAAcArQEAACc2AAAAAA=='
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as archive:
    archive.extractall(task_dir)
print((task_dir/'protocol.json').read_text())


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
               'diffusers==0.39.0', 'transformers==4.57.6',
               'accelerate', 'sentencepiece', 'safetensors', 'pytest'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'test_coupling.py', 'test_capacity_interface.py'],
               cwd=task_dir, check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import datetime, os
drive_root = Path('/content/drive/MyDrive/CEG-WM/discrete-pair-v1')
drive_root.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(drive_root/'model-cache')
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir = drive_root/'runs'/('capacity-canary-'+stamp)
print('Output:', run_dir)
subprocess.run([sys.executable, 'capacity_runner.py', '--count', '2', '--output', str(run_dir)],
               cwd=task_dir, check=True)


In [ ]:
status_path = run_dir/'status.json'
if status_path.exists():
    status = __import__('json').loads(status_path.read_text())
    print(__import__('json').dumps({k:status.get(k) for k in
          ['capacity_summary', 'stop_reason', 'wall_seconds', 'statuses']}, indent=2, ensure_ascii=False))
print('Stop here. Two-source canary is not full development evidence; review the records before expansion.')
